In [ ]:
class G_Unit:
    # Labels
    LIST = 1
    ITEM = 2
    QUOTE = 3
    BREAK = 4
    END = 5
    AND_OR_END = 6
    COLON = 7
    COLON_BREAK = 8
    I_CLAUSE = 9
    D_CLAUSE = 10
    P_PHRASE = 11
    BRACKETS = 12
    FRAGMENT = 13
    CONJ = 14

    def __init__(self, doc, label=None, l=None, r=None, children=None):
        self.doc = doc
        self.label = [] if not label else [label] if not isinstance(label, list) else label
        self.l = l
        self.r = r
        self.children = children or []

    def label_(self):
        labels = []
        if G_Unit.LIST in self.label:
            labels.append("List")
        if G_Unit.ITEM in self.label:
            labels.append("Item")
        if G_Unit.QUOTE in self.label:
            labels.append("Quote")
        if G_Unit.BREAK in self.label:
            labels.append("Break")
        if G_Unit.END in self.label:
            labels.append("End")
        if G_Unit.AND_OR_END in self.label:
            labels.append("And or End")
        if G_Unit.COLON in self.label:
            labels.append("Colon")
        if G_Unit.COLON_BREAK in self.label:
            labels.append("Colon Break")
        if G_Unit.I_CLAUSE in self.label:
            labels.append("Independent Clause")
        if G_Unit.D_CLAUSE in self.label:
            labels.append("Dependent Clause")
        if G_Unit.P_PHRASE in self.label:
            labels.append("Prepositional Phrase")
        if G_Unit.BRACKETS in self.label:
            labels.append("Brackets")
        if G_Unit.FRAGMENT in self.label:
            labels.append("Fragment")
        if G_Unit.CONJ in self.label:
            labels.append("Conjunction")
        return ", ".join(labels) or "None"
        
    def size(self):
        return self.r - self.l + 1

    def span(self):
        return self.doc[self.l:self.r+1]

    def text(self):
        return self.doc[self.l:self.r+1].text
    
    def lower(self):
        return self.doc[self.l:self.r+1].text.lower()

    def start(self):
        return self.doc[self.l]

    def end(self):
        return self.doc[self.r]

    def sent_start(self):
        for sent in self.doc.sents:
            if sent.start <= self.l < sent.end:
                return sent.start
        return -1

    def label_has(self, labels):
        return set(self.label).intersection(labels)

    @staticmethod
    def tokens(*, unit=None, units=None):
        if units:
            tokens = flatten([list(unit.span()) for unit in units])
            tokens = sorted(tokens, key=lambda token: token.i)
            return tokens
        if unit:
            tokens = list(unit.span())
            return tokens
        return None

    @staticmethod
    def is_conjunction(token):
        return token.lower_ in ["and", "or"]

    @staticmethod
    def same_speech(speech_1, speech_2):
        nouns = ["NOUN", "PRON", "PROPN"]
        if speech_1 in nouns and speech_2 in nouns:
            return True
        return speech_1 == speech_2

    @staticmethod
    def same_speech_list(speech_1, speech_2_list):
        for speech_2 in speech_2_list:
            if G_Unit.same_speech(speech_1, speech_2):
                return True
        return False

    def __str__(self):
        return self.text()

In [ ]:
class Quotes:
    def __init__(self, main, units):
        self.main = main
        self.units = units

    def is_quote(self, i):
        return i < len(self.units) and self.units[i].lower() == "\""
    
    def identify(self):
        i = 0
        
        while i < len(self.units):
            if not self.is_quote(i):
                i += 1
                continue
            
            self.units[i].label.append(G_Unit.QUOTE)
            
            while not self.is_quote(i+1):
                self.units[i].r = self.units[i+1].r
                self.units.pop(i+1)

            if self.is_quote(i+1):
                self.units[i].r = self.units[i+1].r
                self.units.pop(i+1)

        return self.units

In [ ]:
class Brackets:
    MATCHES = {
        "[": "]", 
        "(": ")",
        "—": "—",
    }

    OPENING = MATCHES.keys()
    CLOSING = MATCHES.values()

    def __init__(self, main, units):
        self.main = main
        self.stack = []
        self.units = [*units]

    def is_opening(self, i):
        return i < len(self.units) and self.units[i].lower()[0] in Brackets.OPENING

    def is_closing(self, i):
        return i < len(self.units) and self.units[i].lower()[0] in Brackets.CLOSING

    def closes(self, i):
        opener = self.units[self.stack[-1]].lower()[0]
        closer = self.units[i].lower()[0]
        return Brackets.MATCHES[opener] == closer
    
    def identify(self):
        self.stack = []
        
        i = 0
        while i < len(self.units):
            # Closing
            if self.is_closing(i) and self.stack:
                j = None if not self.closes(i) else self.stack.pop()
                
                if not self.stack and j != None:
                    self.units[j].r = self.units[i].r
                    self.units.pop(i)
                    continue
                else:
                    i += 1

            # Opening
            elif self.is_opening(i):
                if not self.stack:
                    self.units[i].label.append(G_Unit.BRACKETS)
                self.stack.append(i)
                i += 1

            # Consuming
            elif self.stack:
                # If you're at the end of the possible units,
                # and the list is unclosed, we must stop.
                if i + 1 >= len(self.units):
                    break
                self.units[self.stack[0]].r = self.units[i+1].r
                self.units.pop(i)

            else:
                i += 1
        
        return self.units

In [ ]:
class Separators:
    def __init__(self, main, units):
        self.main = main
        self.units = [*units]

    def is_break(self, i):
        if i >= len(self.units):
            return False
        
        if self.units[i].lower() not in [";", ","]:
            return False

        # Breaks cannot have a following conjunction.
        # Else, it would be an end and not a break.
        return not bool(
            i + 1 < len(self.units) and 
            self.units[i+1].size() == 1 and 
            self.units[i+1].span()[0].pos_ in ["CCONJ"]
        )

    def is_end(self, i):
        if i >= len(self.units):
            return False
        
        if self.units[i].lower() not in [";", ","]:
            return False
        
        return not self.is_break(i)

    def identify(self):
        i = 0

        while i < len(self.units):
            # Break
            if self.is_break(i):
                self.units[i].label.append(G_Unit.BREAK)
                i += 1

            # End
            elif self.is_end(i):
                conj = self.units[i+1].start().lower_

                if conj in ["and", "or"]:
                    self.units[i].label.append(G_Unit.AND_OR_END)
                else:
                    self.units[i].label.append(G_Unit.END)
                
                self.units[i].r += 1
                self.units.pop(i+1)

            elif self.units[i].start().pos_ == "CCONJ":
                self.units[i].label.append(G_Unit.CONJ)
                i += 1

            else:
                i += 1
                
        return self.units

In [ ]:
class Colons:
    def __init__(self, main, units):
        self.main = main
        self.units = [*units]

    def identify(self):
        i = 0

        while i < len(self.units):
            if self.units[i].lower()[-1] != ":":
                i += 1
                continue

            if not self.units[i].label:
                self.units[i].label.append(G_Unit.COLON_BREAK)

            if i + 1 < len(self.units):
                self.units[i+1].label.append(G_Unit.COLON)
                self.units[i+1].r = self.units[-1].r
                self.units = self.units[:i+2]
            
            break

        return self.units        

In [ ]:
class Independent_Clauses:
    def __init__(self, main, units):
        self.main = main
        self.units = [*units]
        self.allowed = []

    def end(self, i):    
        if i >= len(self.units):
            return True

        if self.units[i].label_has(self.allowed):
            return True
        
        # Here, we check if the unit after
        # the supposed end is a clause. If it
        # is, then we can end at the current unit.
        return bool(
            i + 1 < len(self.units) and 
            self.units[i+1].label_has([
                G_Unit.COLON,
                G_Unit.COLON_BREAK,
                G_Unit.I_CLAUSE,
                G_Unit.D_CLAUSE
            ])
        )

    def identify(self, allowed):
        self.allowed = allowed
        
        i = 0
        
        while i < len(self.units):
            if not self.units[i].label_has(self.allowed):
                i += 1
                continue
            
            # Skip Clause
            if self.units[i].label_has([
                G_Unit.I_CLAUSE, 
                G_Unit.D_CLAUSE, 
                G_Unit.P_PHRASE
            ]):
                i += 1
                continue

            # Create Clause
            self.units[i].label.append(G_Unit.I_CLAUSE)
            while not self.end(i+1):
                self.units[i].r = self.units[i+1].r

                # Add Child
                if self.units[i+1].label_has([G_Unit.BRACKETS, G_Unit.QUOTE, G_Unit.P_PHRASE]):
                    self.units[i].children.append(self.units[i+1])
                    
                self.units.pop(i+1)

            i += 1
            
        return self.units

In [ ]:
class Dependent_Clauses:
    RELATIVE_NOUNS = [
        "who",
        "whom",
        "which",
        "what",
        "that",
        "whose",
        "whomever",
        "whoever",
        "whichever",
        "whatever"
    ]
    
    def __init__(self, main, units):
        self.main = main
        self.units = units
        self.separator = None

    def end(self, i):
        if i >= len(self.units):
            return True

        # Here, we check if the unit after
        # is a clause. As we don't combine two
        # clauses, we must end here if that is
        # the case.
        if bool(
            i + 1 < len(self.units) and 
            self.units[i+1].label_has([
                G_Unit.COLON, 
                G_Unit.COLON_BREAK,
                G_Unit.I_CLAUSE,
                G_Unit.D_CLAUSE
            ])
        ):
            return True

        return bool(
            self.units[i].lower()[0] == self.separator or
            self.units[i].lower() in Dependent_Clauses.RELATIVE_NOUNS or
            self.units[i].start().pos_ in ["SCONJ"]
        )

    def identify(self, separator):
        self.separator = separator
        
        i = 0
        
        while i < len(self.units):
            # Skip
            if self.units[i].label_has([
                G_Unit.COLON,
                G_Unit.COLON_BREAK,
                G_Unit.I_CLAUSE, 
                G_Unit.D_CLAUSE, 
                G_Unit.P_PHRASE
            ]):
                i += 1
                continue

            # Indicators of Dependent Clause
            rel = self.units[i].lower() in Dependent_Clauses.RELATIVE_NOUNS
            sub = self.units[i].start().pos_ == "SCONJ"
            
            if not sub and not rel:
                i += 1
                continue

            # Create Clause
            self.units[i].label.append(G_Unit.D_CLAUSE)
            while not self.end(i+1):
                self.units[i].r = self.units[i+1].r

                # Add Child
                if self.units[i+1].label_has([G_Unit.BRACKETS, G_Unit.QUOTE, G_Unit.P_PHRASE]):
                    self.units[i].children.append(self.units[i+1])
                
                self.units.pop(i+1)

            i += 1
        
        return self.units

In [ ]:
class Prepositional_Phrases:
    
    def __init__(self, main, units):
        self.main = main
        self.units = [*units]

    # A prepositional phrase is typically ended by a noun.
    # Therefore, when we run into a noun, we end the phrase.
    # We must also check that it is the last of the first noun(s)
    # we encounter.
    def last_noun(self, i):
        if bool(
            # 1. End
            i >= len(self.units) or 
            
            # 2. Noun
            self.units[i].start().pos_ not in [
                "NOUN", 
                "PROPN", 
                "PRON"
            ]
        ):
            return False

        return bool(
            i + 1 >= len(self.units) or 
            (
                self.units[i+1].size() == 1 and 
                (
                    self.units[i+1].start().pos_ not in [
                        "NOUN", 
                        "PROPN", 
                        "PRON", 
                        "PART"
                    ] or
                    self.units[i+1].start().lower_ in Dependent_Clauses.RELATIVE_NOUNS
                )
            )
        )
    
    def end(self, i):
        return bool(
            # 1. End of List
            i + 1 >= len(self.units) or
            
            # 2. Clause
            self.units[i+1].label_has([
                G_Unit.COLON,
                G_Unit.COLON_BREAK,
                G_Unit.I_CLAUSE,
                G_Unit.D_CLAUSE,
                G_Unit.P_PHRASE,
                G_Unit.BREAK,
                G_Unit.AND_OR_END,
                G_Unit.END
            ]) or
            
            # 3. Noun
            self.last_noun(i)
        )

    def identify(self):    
        i = 0
        
        while i < len(self.units):
            # Skip
            is_comp = self.units[i].size() != 1
            is_non_adp = self.units[i].start().pos_ != "ADP"
            is_not_to = self.units[i].lower() != "to"
            
            if (is_comp or is_non_adp) and is_not_to:
                i += 1
                continue

            # Create Clause
            self.units[i].label.append(G_Unit.P_PHRASE)
            noun_seen = False
            
            while not self.end(i+1):
                noun_seen = noun_seen or self.units[i+1].start().pos_ in [
                    "NOUN", 
                    "PROPN", 
                    "PRON"
                ]
                # Add Child
                if self.units[i+1].label_has([G_Unit.BRACKETS, G_Unit.QUOTE]):
                    if noun_seen:
                        break
                    self.units[i].children.append(self.units[i+1])
                
                self.units[i].r = self.units[i+1].r
                self.units.pop(i+1)

            if bool(
                    self.units[i+1].size() == 1 and 
                    self.units[i+1].span()[0].pos_ in ["NOUN", "PROPN", "PRON", "ADJ", "VERB"] and 
                    self.units[i+1].lower() not in Dependent_Clauses.RELATIVE_NOUNS
                ):
                self.units[i].r = self.units[i+1].r
                self.units.pop(i+1)
            
            i += 1
        
        return self.units   

In [ ]:
class Lists:
    NOUNS = ["NOUN", "PRON", "PROPN"]


    
    def __init__(self, main, units, enclosures):
        self.main = main
        self.units = [*units]
        self.separator = None
        self.enclosures = enclosures


    
    def is_stop(self, unit):
        is_break = G_Unit.BREAK in unit.label and unit.lower()[0] == self.separator
        is_clause = unit.label_has([
            G_Unit.I_CLAUSE, 
            G_Unit.D_CLAUSE, 
            G_Unit.P_PHRASE,
            G_Unit.COLON,
            G_Unit.COLON_BREAK
        ])
        return is_break or is_clause


    
    def find_lists(self, sep):
        self.separator = sep
        
        lists = [
            [
                [None, None]
            ]
        ]

        # print(f"find_lists")
        # print(f"Units:")
        # for i, unit in enumerate(self.units):
        #     print(f"{i}, {unit}")
        
        i = 0
        stack = []
        enclosures = {
            "[": "]",
            "{": "}",
            "(": ")",
            "\"": "\""
        }

        stack_pass = False
        text = " ".join([unit.lower() for unit in self.units])
        opening_characters = [char for char in text if char in enclosures.keys()]
        closing_characters = [char for char in text if char in enclosures.values()]

        if len(opening_characters) == 1 and len(closing_characters) == 1:
            if text[0] in opening_characters and text[-1] in closing_characters:
                if enclosures[text[0]] == text[-1]:
                    stack_pass = True

        # print(f"Text: {text}")
        # print(f"Stack Pass: {stack_pass}")
        
        while i < len(self.units):
            unit = self.units[i]

            if not stack_pass:
                if unit.lower() in list(enclosures.keys()):
                    stack.append(unit.lower())
                if stack and unit.lower() in list(enclosures.values()) and enclosures[stack[-1]] == unit.lower():
                    stack.pop()
            
            # num_conj = 0
            # if not unit.label_has([Unit.AND_OR_END]):
            #     for token in unit.span():
            #         if token.pos_ == "CCONJ" or token.lower_ == sep:
            #             num_conj += 1

            opened = lists[-1][0] != [None, None]
            close_list = unit.label_has([G_Unit.AND_OR_END]) and unit.lower()[0] == sep
            close_item = unit.label_has([G_Unit.BREAK]) and unit.lower() == sep
            remove_list = unit.label_has([G_Unit.COLON, G_Unit.COLON_BREAK])
        
            # Close List
            if not stack and opened and close_list:
                # Invalid List, Remove
                if len(lists[-1]) < 2:
                    lists[-1] = [[None, None]]
                    i += 1
                    continue
                    
                # Find the L Index of Last Item
                last_item_l = i + 1

                # Find the R Index of Last Item
                last_item_r = last_item_l
                
                length = find_index(self.units[last_item_l:], lambda e: self.is_stop(e))
                if length > 0:
                    last_item_r += length - 1
                elif length == -1:
                    last_item_r = len(self.units) - 1

                # Add Last Item
                lists[-1].append([last_item_l, last_item_r])
                lists.append([[None, None]])
                i += 1

            # Close Item
            elif not stack and opened and close_item:
                lists[-1].append([i + 1, i])
                i += 1
                
            # Remove List
            elif not stack and opened and remove_list:
                lists[-1] = [[None, None]]
                i += 1
            
            # Continue Item
            else:
                if not opened:
                    lists[-1][0] = [i, i]
                else:
                    lists[-1][-1][1] += 1
                i += 1
            
        # print(f"1. Lists: {lists}")
        # If we reach the end of the list and the last
        # list is invalid (< 3 items), we remove it.
        if bool(
            lists and len(lists[-1]) < 3 or 
            (
                lists and
                not find(self.units[lists[-1][0][0]:], lambda e: e.label_has([G_Unit.AND_OR_END]) and e.lower()[0] == sep)
            )
        ):
            lists.pop()
        # print(f"2. Lists: {lists}")
        # In each item, we look for pairs (e.g. X and Y).
        # We only handle one conjunction.
        num_lists = len(lists)
        for i, lst in enumerate(lists):
            if i >= num_lists:
                break
            
            for l, r in lst:
                tokens = G_Unit.tokens(units=self.units[l:r+1])
                conj = find_all(tokens, lambda t: G_Unit.is_conjunction(t))
                if len(conj) == 1:
                    lists.append([[l, r]])
        # print(f"3. Lists: {lists}")
        # If there's no lists at all, we can take advantage
        # of lax rules.
        if not lists:
            stack = []
            lst = [[None, None]]
            i = 0
            conj_seen = False
            while i < len(self.units):
                unit = self.units[i]

                if unit.lower() in list(enclosures.keys()):
                    stack.append(unit.lower())
    
                if stack and unit.lower() in list(enclosures.values()) and enclosures[stack[-1]] == unit.lower():
                    stack.pop()
                    
                # num_conj = 0
                # # print("HERE")
                # if not self.units[i].label_has([Unit.AND_OR_END]):
                #     # print(self.units[i])
                #     for token in self.units[i].span():
                #         if token.pos_ == "CCONJ" or token.lower_ == sep:
                #             num_conj += 1
                #     # print(num_conj)

                # if num_conj > 1:
                #     lst = [[None, None]]
                #     if lists:
                #         lists.pop()
                
                if unit.label_has([G_Unit.CONJ]):
                    if not conj_seen:
                        conj_seen = True
                    elif lst != [[None, None]]:
                        lst[-1][1] = i - 1
                        lists.append(lst)
                        lst = [[i - 1, i]]
                # elif self.units[i].label_has([Unit.BREAK, Unit.AND_OR_END, Unit.END]) and self.units[i].lower() == sep:
                #     if lst != [[None, None]]:
                #         lists.append(lst)
                #     lst = [[None, None]]
                else:
                    if lst == [[None, None]]:
                        lst = [[i, i]]
                    else:
                        lst[-1][1] = i
                
                i += 1
            
            if lst != [[None, None]] and conj_seen:
                lists.append(lst)
        # print(f"4. Lists: {lists}")
        # Here we remove duplicates, I'm not sure if duplicates still
        # occur, I observed them once, but this is here in case.
        # Note: I could do a cheeky list(set(...)), at least I think.
        i = 0
        while i < len(lists):
            if lists[i] in lists[i+1:]:
                lists.pop(i)
            else:
                i += 1
        # print(f"5. Lists: {lists}")
        # Remove Invalid Lists
        i = 0
        while i < len(lists):
            # The list contains one item and that item only contains one
            # token, or the list has two items.
            if bool(
                (
                    len(lists[i]) == 1 and 
                    lists[i][0][0] == lists[i][0][1]
                ) or
                (
                    len(lists[i]) == 1 and
                    # An AND/OR is required.
                    not [unit for unit in self.units[lists[i][0][0]:lists[i][0][1]+1] if "and" in unit.lower() or "or" in unit.lower()]
                ) or
                len(lists[i]) == 2
            ):
                lists.pop(i)
            else:
                i += 1
        # print(f"6. Lists: {lists}")
        return lists


    
    def clean_lists(self, lists):
        overlaps = []

        i = 0
        while i + 1 < len(lists):
            a = lists[i]
            b = lists[i+1]
                  
            if a[-1] != b[0]:
                i += 1
                continue

            if len(a) <= 1 or len(b) <= 1:
                i += 1
                continue

            # No Way to Split
            if a[-1][1] - a[-1][0] == 0 and b[0][1] - b[0][0] == 0:
                overlaps.extend([i, i + 1])
                i += 2
            else:
                a[-1][1] = a[-1][0]
                b[0][0] = b[0][1]
                i += 2

        lists = [l for i, l in enumerate(lists) if i not in overlaps]
        # print(f"7. Lists: {lists}")
        return lists


    
    def expand_noun(self, tokens, start, direction):
        for group in [*self.main.sp_doc.noun_chunks, *self.main.sp_doc.ents]:
            tokens_i = [t.i for t in group]
            if tokens[start].i in tokens_i:
                while start >= 0 and start < len(tokens) and tokens[start].i in tokens_i:
                    start += 1 * direction
                start += 1 * direction * -1
                break
        
        return start


    
    def char_bound_list(self, lst):
        # We bound each item according to characters or a speech.
        # We find these bounds from the "base item", the second to last item.
        base_tokens = G_Unit.tokens(units=self.units[lst[-2][0]:lst[-2][1]+1])
        
        # As we're bounding by characters, primarily, the left bound is just
        # the characters of the first token
        l_bound = base_tokens[0].lower_

        # The right bound is the first tag, of the below set of tags, that we
        # encounter in the base tokens. If there's not such a token, we cannot
        # bound the items.
        speech = ["NOUN", "PROPN", "PRON", "VERB", "NUM"]
        r_bound = None
        for i in range(len(base_tokens) - 1, -1, -1):
            if base_tokens[i].pos_ in speech:
                r_bound = base_tokens[i]
                break

        if not r_bound:
            return None

        # The inner items are already bounded on the left and right sides.
        # All we need to check is whether the start matches with the left bound.
        inner_items = lst[1:-2]

        for i, item in enumerate(inner_items):
            l = item[0]
            r = item[1]
            
            tokens = G_Unit.tokens(units=self.units[l:r+1])

            # If it doesn't match, we check if the next set of items can be
            # bounded. If not, we cannot bound the list.
            if tokens[0].lower_ != l_bound:
                if len(inner_items) - i - 1 >= 2:
                    return self.bound_list(lst[i+2:])
                return None
            
        # Check for L Bound in Starting Item
        start_tokens = G_Unit.tokens(units=self.units[lst[0][0]:lst[0][1]+1])
        start_l = len(start_tokens) - 1
        while start_l >= 0 and start_tokens[start_l].lower_ != l_bound:
            start_l -= 1

        # L Bound Not Found
        if start_l < 0:
            # If the list is greater than 4 items, we can
            # cut off the starting item, and try again.
            if len(inner_items) >= 2:
                return self.bound_list(lst[1:])
            return None

        # If the first of the start tokens is a noun, there may be more
        # to include.
        if start_tokens[start_l].pos_ in Lists.NOUNS:
            start_l = self.expand_noun(start_tokens, start_l, -1)
                    
        # Check for R Bound in Ending Item
        end_tokens = G_Unit.tokens(units=self.units[lst[-1][0]:lst[-1][1]+1])
        end_r = 0
        num_end_tokens = len(end_tokens)
        while end_r < num_end_tokens and end_tokens[end_r].pos_ not in speech:
            end_r += 1

        if end_r >= num_end_tokens:
            return None

        # If the last of the end tokens is a noun, there may be more
        # to include.
        if end_tokens[end_r].pos_ in Lists.NOUNS:
            end_r = self.expand_noun(end_tokens, end_r, 1)
        
        # Create List
        unit_start_item = G_Unit(self.main.sp_doc, label=G_Unit.ITEM, l=start_tokens[start_l].i, r=start_tokens[-1].i)
        unit_end_item = G_Unit(self.main.sp_doc, label=G_Unit.ITEM, l=end_tokens[0].i, r=end_tokens[end_r].i)
        
        unit_list = G_Unit(self.main.sp_doc, label=G_Unit.LIST, l=start_tokens[start_l].i, r=end_tokens[end_r].i)
        unit_list.children.extend([unit_start_item, unit_end_item])
        
        for item in lst[1:-1]:
            tokens = G_Unit.tokens(units=self.units[item[0]:item[1]+1])
            unit_item = G_Unit(self.main.sp_doc, label=G_Unit.ITEM, l=tokens[0].i, r=tokens[-1].i)
            unit_list.children.append(unit_item)

        return unit_list


    
    def char_bound_pair(self, pair):
        tokens = G_Unit.tokens(units=self.units[pair[0][0]:pair[0][1]+1])
        tokens = sorted(tokens, key=lambda t: t.i)
        num_tokens = len(tokens)
        
        m = find_index(tokens, lambda t: G_Unit.is_conjunction(t))

        l = m - 1
        r = m + 1

        # Bound L by R Token Characters
        i = m - 1
        while i >= 0 and tokens[i].lower_ != tokens[m + 1].lower_:
            i -= 1

        if i < 0:
            return None

        # Bound R by L Token Speech
        j =  m + 1
        while j < num_tokens and not G_Unit.same_speech(tokens[m-1].pos_, tokens[j].pos_):
            j += 1

        if j >= num_tokens:
            return None
        
        e_item_l = G_Unit(self.main.sp_doc, label=G_Unit.ITEM, l=tokens[i].i, r=tokens[m-1].i)
        e_item_r = G_Unit(self.main.sp_doc, label=G_Unit.ITEM, l=tokens[m+1].i, r=tokens[j].i)
        e_list = G_Unit(self.main.sp_doc, label=G_Unit.LIST, l=tokens[i].i, r=tokens[j].i, children=[e_item_l, e_item_r])
        return e_list


    
    def bound_list(self, lst):
        # Base Item (2nd to Last Item) Tokens
        # This item is already bounded by the
        # left and right sides, which is useful.
        base_tokens = G_Unit.tokens(units=self.units[lst[-2][0]:lst[-2][1]+1])
        num_base_tokens = len(base_tokens)
        
        # Speech Bounds
        speech = ["NOUN", "PROPN", "PRON", "VERB"]
        adjectives = ["ADJ", "ADV", "NUM", "ADP", "AUX"]
        
        # Find L Bound
        l_bound = []
        for i in range(0, num_base_tokens):
            if base_tokens[i].pos_ in speech:
                l_bound = [base_tokens[i].pos_]
                break
            elif base_tokens[i].pos_ in adjectives:
                l_bound = [base_tokens[i].pos_]

                j = i + 1
                while j < num_base_tokens:
                    if base_tokens[j].pos_ in speech:
                        l_bound.append(base_tokens[j].pos_)
                        break
                    j += 1
                
                break
        
        if not l_bound:
            return None

        
        # Find R Bound
        r_bound = []
        for i in range(num_base_tokens - 1, -1, -1):
            if base_tokens[i].pos_ in speech:
                r_bound = [base_tokens[i].pos_]
                break
            elif base_tokens[i].pos_ in adjectives:
                r_bound = [base_tokens[i].pos_]

                j = i - 1
                while j >= 0:
                    if base_tokens[j].pos_ in speech:
                        r_bound.append(base_tokens[j].pos_)
                        break
                    j -= 1
                
                break

        if not r_bound:
            return None
        
        # Check Inner Items
        # The inner items must have the left bound,
        # the right bound isn't as important.
        inner_items = lst[1:-1]

        verb_seen = False
        for i, item in enumerate(inner_items):
            l = item[0]
            r = item[1]
            
            item_tokens = G_Unit.tokens(units=self.units[l:r+1])
            item_speech = [token.pos_ for token in item_tokens]

            # Must be Homogeneous
            if "VERB" not in item_speech and verb_seen:
                if len(inner_items) >= 2:
                    return self.bound_list(lst[1:])  
                else:
                    return None
            elif "VERB" in item_speech:
                verb_seen = True

            # Not Found
            if not set(l_bound).intersection(item_speech):
                # We check if the list starting at the next
                # item has a chance. If it does, that becomes
                # the list.
                if len(inner_items) - i + 1 >= 2:
                    return self.bound_list(lst[i+2:])
                return None
        
        # Check Starting Item
        start_tokens = G_Unit.tokens(units=self.units[lst[0][0]:lst[0][1]+1])
        start_l = len(start_tokens) - 1
        
        while start_l >= 0 and not G_Unit.same_speech_list(start_tokens[start_l].pos_, l_bound):
            start_l -= 1

        if start_l < 0:
            if len(inner_items) >= 2:
                return self.bound_list(lst[1:])
            return None

        # Adjust Starting Item
        if set(l_bound).intersection(["ADJ", "NUM", "ADV", *Lists.NOUNS]):
            start_l = self.expand_noun(start_tokens, start_l, -1)
        
        # Check Ending Item
        end_tokens = G_Unit.tokens(units=self.units[lst[-1][0]:lst[-1][1]+1])
        end_r = 0
        num_end_tokens = len(end_tokens)

        while end_r < num_end_tokens and not G_Unit.same_speech_list(end_tokens[end_r].pos_, r_bound):
            end_r += 1

        if end_r >= num_end_tokens:
            return None

        # Adjust Ending Item
        if set(r_bound).intersection(["ADJ", "NUM", "ADV", *Lists.NOUNS]):
            end_r = self.expand_noun(end_tokens, end_r, 1)
    
        # Adjusting Bounds for Start and End Entities
        l_i = start_tokens[start_l].i
        l_label = [G_Unit.ITEM]
        
        r_i = end_tokens[end_r].i
        r_label = [G_Unit.ITEM]

        # Handle Potential Overlap Issues w Enclosures
        for ent in self.enclosures:
            if not ent.label_has([G_Unit.BRACKETS, G_Unit.QUOTE]):
                continue
            
            l_overlap = ent.l <= start_tokens[start_l].i <= ent.r
            r_overlap = ent.l <= end_tokens[end_r].i <= ent.r

            # Left Item
            if l_overlap and not r_overlap:
                l_label.extend(list(set(ent.label) & set([G_Unit.BRACKETS, G_Unit.QUOTE])))
                l_i = min(ent.l, l_i)

            # Right Item
            if not l_overlap and r_overlap:
                l_label.extend(list(set(ent.label) & set([G_Unit.BRACKETS, G_Unit.QUOTE])))
                r_i = max(ent.r, r_i)
      
        unit_list = G_Unit(self.main.sp_doc, label=G_Unit.LIST, l=l_i, r=r_i)

        unit_start_item = G_Unit(self.main.sp_doc, label=l_label, l=l_i, r=start_tokens[-1].i)
        unit_end_item = G_Unit(self.main.sp_doc, label=r_label, l=end_tokens[0].i, r=r_i)
        unit_list.children.extend([unit_start_item, unit_end_item])

        for item in lst[1:-1]:
            tokens = G_Unit.tokens(units=self.units[item[0]:item[1]+1])
            unit_item = G_Unit(self.main.sp_doc, label=G_Unit.ITEM, l=tokens[0].i, r=tokens[-1].i)
            unit_list.children.append(unit_item)

        return unit_list


    
    def bound_pair(self, pair):
        tokens = G_Unit.tokens(units=self.units[pair[0][0]:pair[0][1]+1])
        tokens = sorted(tokens, key=lambda t: t.i)
        num_tokens = len(tokens)
        
        # Conjunction Partitions
        m = find_index(tokens, lambda t: G_Unit.is_conjunction(t))
        m_i = tokens[m].i

        # Speech for Bounding
        # We handle lists of the types below.
        speech = ["NOUN", "PROPN", "PRON", "VERB"]
        adjectives = ["ADJ", "ADV", "NUM", "ADP", "AUX"]

        # Find L Bound
        l_bound = []
        l_bound_i = None
        
        for i in range(m + 1, num_tokens):
            if tokens[i].pos_ in speech:
                l_bound = [tokens[i].pos_]
                l_bound_i = tokens[i].i
                break
            # With adjectives, we can also add the following token
            # as a bound. This allows a list like "X and [ADJ] Y"
            # to be recognized.
            elif tokens[i].pos_ in adjectives:
                l_bound = [tokens[i].pos_]

                j = i + 1
                while j < num_tokens:
                    if tokens[j].pos_ in speech:
                        l_bound.append(tokens[j].pos_)
                        break
                    j += 1
                
                break

        if not l_bound:
            return None
        
        # Find R Bound
        r_bound = []
        r_bound_i = None
        
        for i in range(m - 1, -1, -1):
            if tokens[i].pos_ in speech:
                r_bound = [tokens[i].pos_]
                r_bound_i = tokens[i].i
                break
            # With adjectives, we can also list the following token
            # as a bound. This allows a list like "X and [ADJ] Y"
            # to be recognized.
            elif tokens[i].pos_ in adjectives:
                r_bound = [tokens[i].pos_]

                j = i - 1
                while j >= 0:
                    if tokens[j].pos_ in speech:
                        r_bound.append(tokens[j].pos_)
                        break
                    j -= 1
                
                break
        
        if not r_bound:
            return None
        
        # Bound L Item
        l = m - 1
        while l >= 0 and not G_Unit.same_speech_list(tokens[l].pos_, l_bound):
            l -= 1

        if l < 0:
            return None

        # Adjust L if Noun
        if l_bound in Lists.NOUNS:
            l = self.expand_noun(tokens, l, -1)
        
        # Bound R Item
        r = m + 1
        while r < num_tokens and not G_Unit.same_speech_list(tokens[r].pos_, r_bound):
            r += 1
        
        if r >= num_tokens:
            return None

        # Adjust R if Noun
        if r_bound in Lists.NOUNS:
            r = self.expand_noun(tokens, r, 1)

        # Further Adjusting Bounds for Entities
        l_i = tokens[l].i
        l_label = [G_Unit.ITEM]
        
        r_i = tokens[r].i
        r_label = [G_Unit.ITEM]
        for ent in self.enclosures:
            if not ent.label_has([G_Unit.BRACKETS, G_Unit.QUOTE]):
                continue
            
            l_overlap = ent.l <= tokens[l].i <= ent.r
            r_overlap = ent.l <= tokens[r].i <= ent.r

            # Left Item
            if l_overlap and not r_overlap:
                l_label.extend(list(set(ent.label) & set([G_Unit.BRACKETS, G_Unit.QUOTE])))
                l_i = min(ent.l, l_i)

            # Right Item
            if not l_overlap and r_overlap:
                l_label.extend(list(set(ent.label) & set([G_Unit.BRACKETS, G_Unit.QUOTE])))
                r_i = max(ent.r, r_i)

        e_item_l = G_Unit(self.main.sp_doc, label=l_label, l=l_i, r=m_i-1)
        e_item_r = G_Unit(self.main.sp_doc, label=r_label, l=m_i+1, r=r_i)
        e_list = G_Unit(self.main.sp_doc, label=G_Unit.LIST, l=l_i, r=r_i)
        e_list.children.extend([e_item_l, e_item_r])
        
        return e_list


    
    def bound_lists(self, lists):
        bound_lists = []
        
        for lst in lists:
            bound = None
        
            if len(lst) == 1:
                bound = self.char_bound_pair(lst)
                if not bound:
                    bound = self.bound_pair(lst)
            else:
                bound = self.char_bound_list(lst)
                if not bound:
                    bound = self.bound_list(lst)
            
            if bound:
                bound_lists.append(bound)
        # print(f"8. Lists:")
        # for bound_list in bound_lists:
        #     print(f"-> {bound_list}")
        return bound_lists


    
    def merge_lists(self, bound_lists):
        # Map (L, R) to Unit List
        mapped_bounds = {}
        for lst in bound_lists:
            mapped_bounds[(lst.l, lst.r)] = lst
        bounds = list(mapped_bounds.keys())

        # Find Largest Coverage of Bounds
        max_coverage = []
        
        for bound in bounds:
            overlap = False
            for i, max_bound in enumerate(max_coverage):
                contains = max_bound[0] <= bound[0] <= max_bound[1] or max_bound[0] <= bound[1] <= max_bound[1]
                surround = bound[0] <= max_bound[0] <= bound[1] or bound[0] <= max_bound[1] <= bound[1]
                
                if contains or surround:
                    overlap = True
                
                    if bound[1] - bound[0] > max_bound[1] - max_bound[0]:
                        max_coverage[i] = bound
            
            if not overlap:
                max_coverage.append(bound)
        
        # Integrate Lists
        for bound in max_coverage:
            l_overlap = None
            l_overlap_i = None
            
            r_overlap = None
            r_overlap_i = None
            
            i = 0
            while i < len(self.units):
                unit = self.units[i]
                
                # Overlap w/ Left
                if not l_overlap and unit.l <= bound[0] <= unit.r:
                    l_overlap = unit
                    l_overlap_i = i
    
                # Overlap w/ Right
                if unit.l <= bound[1] <= unit.r:
                    r_overlap = unit
                    r_overlap_i = i

                if l_overlap and r_overlap:
                    break

                i += 1

            if l_overlap.label_has([G_Unit.BRACKETS, G_Unit.QUOTE]):
                self.units = self.units[:l_overlap_i] + self.units[r_overlap_i+1:]
                self.units.insert(l_overlap_i, mapped_bounds[bound])
                
                mapped_bounds[bound].l = min(l_overlap.l, mapped_bounds[bound].l)
                mapped_bounds[bound].r = max(l_overlap.r, mapped_bounds[bound].r)
                
            elif l_overlap.label_has([G_Unit.I_CLAUSE, G_Unit.D_CLAUSE, G_Unit.P_PHRASE]):
                if l_overlap.l == mapped_bounds[bound].l:
                    # Add Children
                    l_overlap.r = max(l_overlap.r, mapped_bounds[bound].r)
                    l_overlap.children.append(mapped_bounds[bound])
                    self.units = self.units[:l_overlap_i+1] + self.units[r_overlap_i+1:]
                else:
                    # Add Children
                    l_overlap.r = max(l_overlap.r, mapped_bounds[bound].r)
                    l_overlap.children.append(mapped_bounds[bound])
                    self.units = self.units[:l_overlap_i+1] + self.units[r_overlap_i+1:]
                    
            else:
                self.units = self.units[:l_overlap_i] + self.units[r_overlap_i+1:]
                self.units.insert(l_overlap_i, mapped_bounds[bound])

        return self.units
        
    def identify(self, sep):
        lists = self.find_lists(sep)
        lists = self.clean_lists(lists)
        lists = self.bound_lists(lists)
        lists = self.merge_lists(lists)
        return lists

In [ ]:
class Units:
    def __init__(self, main):
        self.main = main
        self.unit_map = {}
        
    def update(self):
        self.unit_map = self.load_full_unit_map()

    def load_full_unit_map(self):
        unit_map = {}
        
        for sent in self.main.sp_doc.sents:
            sent_tokens = list(sent)
            sent_unit_map = self.load_units(sent_tokens)
            unit_map.update(sent_unit_map)
        
        return unit_map

    def load_unit_map(self, ent):
        ent_map = {}
        if ent.l != -1:
            ent_map[(ent.l, ent.r)] = ent

        # Add Children
        for child in ent.children:
            child_ent_map = self.load_unit_map(child)
            ent_map.update(child_ent_map)
        
        return ent_map
    
    def load_units(self, tokens, load_clauses=True, load_enclosures=True):
        # print(f"Tokens: {tokens}")
        
        units = []
        for token in tokens:
            unit = G_Unit(
                self.main.sp_doc, 
                l=token.i, 
                r=token.i
            )
            units.append(unit)

        # Enclosures
        # These are extracted first due to their
        # simplicity.
        if load_enclosures:
            units = Quotes(self.main, units).identify()
            units = Brackets(self.main, units).identify()

        # These class of units can be put inside a larger
        # unit (which makes it hard to know where they are
        # later on). Thus, they're stored in this variable for
        # later use in the list identification.
        enclosures = []
        for unit in units:
            if unit.label_has([G_Unit.BRACKETS, G_Unit.QUOTE]):
                enclosures.append(unit)
        
        # Find Partioning Separator
        # If a sentence uses the below structure:
        # "When ... , .... ; therefore, ... ."
        # There is a hierarchical structure between
        # the semicolons and commas where the former
        # nests the latter. Therefore, we find the
        # higher-ranking separator (which would be
        # the semicolon, if there's any) to use for
        # further separation.
        sep = ","
        for unit in units:
            if ";" == unit.lower()[0]:
                sep = ";"
                break

        # Separators and Colons
        # The separators here also include conjunctions and
        # the use of both conjunctions and punctuation.
        units = Separators(self.main, units).identify()

        if load_enclosures:
            units = Colons(self.main, units).identify()

        if load_clauses:
            units = Prepositional_Phrases(self.main, units).identify()
            units = Dependent_Clauses(self.main, units).identify(sep)
            units = Independent_Clauses(self.main, units).identify([G_Unit.END])
        
        units = Lists(self.main, units, enclosures).identify(sep)
        
        # There is some overlap between lists and independent
        # clauses because they both can use ", [AND/OR]", but
        # after the lists are identified, we can assume the
        # remaining ", [AND/OR]" are parts of independent 
        # clauses.
        if load_clauses:
            units = Independent_Clauses(self.main, units).identify([G_Unit.AND_OR_END])

        # Merge Ungrouped Entities
        i = 0
        while i < len(units):
            if not units[i].label:
                while i + 1 < len(units) and (not units[i+1].lower() in string.punctuation and (not units[i+1].label)):
                    units.pop(i+1)
                    units[i].r += 1
                units[i].label = [G_Unit.FRAGMENT]
            i += 1

        # Remove Fragments
        # If we're already parsing a fragment
        # (indicated by load_clauses = False), we
        # should not add meaningless, duplicate fragments.
        if not load_clauses or not load_enclosures:
            units = [ent for ent in units if ent.label != [G_Unit.FRAGMENT]]
        
        # Map Entities
        # This map lists the units, the units' children,
        # those childrens' children, and so on, in a convenient
        # manner, arguably. It all starts with one unit, which
        # we have as the "parent". It encapsulates the units
        # we want to list.
        parent = G_Unit(self.main.sp_doc, l=-1, r=-1, children=units)
        parent_ent_map = self.load_unit_map(parent)
        
        for unit in list(set([*units, *enclosures])):
            unit_tokens = [*unit.span()]

            visitable = unit.label_has([
                G_Unit.I_CLAUSE, 
                G_Unit.D_CLAUSE, 
                G_Unit.FRAGMENT,
                G_Unit.COLON,
                G_Unit.BRACKETS,
                G_Unit.QUOTE
            ])
            non_empty_subset = 2 < len(unit_tokens) < len(tokens)
            
            if not non_empty_subset or not visitable:
                continue

            load_clauses = not unit.label_has([G_Unit.I_CLAUSE, G_Unit.D_CLAUSE, G_Unit.FRAGMENT])
            load_enclosures = not unit.label_has([G_Unit.COLON, G_Unit.QUOTE, G_Unit.BRACKETS])
            
            unit_map = self.load_units(unit_tokens, load_clauses=load_clauses, load_enclosures=False)
            for k, v in unit_map.items():
                parent_ent_map[k] = v
            
        return parent_ent_map

    def units_at_i(self, i):
        units = []
        for k, v in self.unit_map.items():
            if k[0] <= i <= k[1]:
                units.append(v)
        return units

    def aggregate_units(self, aggregate_labels=[G_Unit.LIST]):
        unit_bounds = [k for k, v in self.unit_map.items() if not v.label_has([G_Unit.ITEM])]
        distinct_unit_bounds = distinct_bounds(unit_bounds, larger=False)
        
        units = [unit[1] for unit in self.unit_map.items() if unit[0] in distinct_unit_bounds]
        units = sorted(units, key=lambda unit: unit.l)
        
        i = 0
        tokens = []
        units_tokens = []
        
        while i < len(units):            
            unit = units[i]
            print(unit)
            tokens.extend([*unit.span()])
            next_unit = None if i+1 >= len(units) else units[i+1]
            
            if next_unit and next_unit.label_has(aggregate_labels):
                i += 1
                continue

            if i and units_tokens:
                l = units_tokens[-1][-1].i
                r = tokens[0].i
                if r - l > 1:
                    units_tokens[-1].extend(list(self.main.sp_doc[l+1:r]))
            
            units_tokens.append(tokens)
            tokens = []
            
            i += 1

        if tokens:
            units_tokens.append(tokens)

        return units_tokens

    def aggregate_tokens_by_unit(self):
        split_tokens = aggregate_tokens_by_unit(self.main, self)
        return split_tokens

In [ ]:
def a_contains_b(a, b):
    return (a[0] <= b[0] < a[1] and a[0] <= b[1] < a[1]) or (a[0] < b[0] <= a[1] and a[0] < b[1] <= a[1])

def a_overlaps_b(a, b):
    return a[0] <= b[0] <= a[1] or a[0] <= b[1] <= a[1]

def aggregate_tokens_by_unit(main, units):
    # Split Tokens
    split_tokens = []
    tokens = [] # Running List of Tokens
    
    i = 0
    num_tokens = len(main.sp_doc)
    
    while i < num_tokens:
        token = main.sp_doc[i]

        # Case 1: End of Sentence (.)
        end_of_sentence_1 = token.i == token.sent.end - 1 and token.pos_ == "PUNCT"
        end_of_sentence_2 = i + 1 < num_tokens and main.sp_doc[i+1].sent.start != token.sent.start
        if end_of_sentence_1 or end_of_sentence_2:
            tokens and split_tokens.append(tokens)
            tokens = []
        # Case 2: Comma (,), Semicolon (;), Colon (:), Conjunction (AND/OR)
        elif token.lower_[0] in ",;:" or token.pos_ == "CCONJ":
            t_units = units.units_at_i(token.i)
            unit_lists = [unit for unit in t_units if unit.label_has([G_Unit.LIST])]

            if unit_lists:
                tokens.append(token)
                i += 1
            else:
                unit_breaks = [unit for unit in t_units if unit.label_ in [G_Unit.AND_OR_END, G_Unit.END, G_Unit.BREAK, G_Unit.COLON_BREAK, G_Unit.CONJ]]
                tokens and split_tokens.append(tokens)
                tokens = []
    
                if unit_breaks:
                    unit = unit_breaks[0]
                    i += unit.r - unit.l + 1
                else:
                    i += 1
            
            continue
        # Case 3: Regular Token
        else:
            t_units = units.units_at_i(token.i)

            split = False
            cont_loop = False
            
            for unit in t_units:
                if unit.label_has([G_Unit.I_CLAUSE, G_Unit.D_CLAUSE]) and token.i == unit.l:
                    tokens and split_tokens.append(tokens)
                    tokens = [token]
                    split = True
                elif unit.label_has([G_Unit.BRACKETS]):
                    i += unit.r - unit.l + 1
                    split_tokens.append([*unit.span()])
                    split = True
                    cont_loop = True
                elif unit.label_has([G_Unit.P_PHRASE]) and (unit.start().pos_ == "SCONJ" or not tokens):
                    i += unit.r - unit.l + 1
                    split_tokens.append([*unit.span()])
                    split = True
                    cont_loop
            
            # We can't continue the while-loop
            # from the for-loop.
            if cont_loop:
                continue

            if not split:
                tokens.append(token)
        
        i += 1
    
    if tokens:
        split_tokens.append(tokens)

    return split_tokens

In [132]:
# # import spacy
# # import string
# # %run "Helper.ipynb"

# # class Main:
# #     def __init__(self):
# #         self.sp_nlp = spacy.load("en_core_web_lg")
# #         self.sp_doc = None

# # main = Main()
# main.sp_doc = main.sp_nlp("The cat caused the dog to cry which made the bird increase in size")

# units = Units(main)
# units.update()
# unit_map_bounds = units.unit_map.keys()
# unit_map_bounds = sorted(unit_map_bounds)

# for bound in unit_map_bounds:
#     unit = units.unit_map[bound]
#     print(f"({unit.l}, {unit.r}) ({unit.label_()}) -> {main.sp_doc[unit.l:unit.r+1]}")

# split_tokens = units.aggregate_tokens_by_unit()
# for tokens in split_tokens:
#     print(tokens)